# Aula 11 - Notebook: Modelagem Topológica de Tubulações como Dígrafos Ponderados

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Linha de Produção de Paçoca  
**Equipe:** Grupo 7  
**Perfil:** Engenharia de Controle e Automação (Matemática Discreta & Teoria dos Grafos)  

---

Neste notebook implementamos a classe base `GrafoTubulacao` para representar a malha física de transporte de sólidos, moagem, homogeneização e prensagem da **Fábrica de Paçoca** como um Grafo Dirigido e Ponderado $G=(V, E, W)$.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from typing import List, Dict, Tuple, Any

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float, 
                           tag_valvula: str, diametro_pol: float = 4.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula,
            "Diâmetro (pol)": diametro_pol
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({"Equipamento": v, "Grau Entrada (deg-)": deg_in, "Grau Saída (deg+)": deg_out})
        return graus

    def verificar_aperto_maos(self) -> Dict[str, Any]:
        graus = self.obter_graus()
        soma_in = sum(g["Grau Entrada (deg-)"] for g in graus)
        soma_out = sum(g["Grau Saída (deg+)"] for g in graus)
        total_e = len(self.arestas_detalhes)
        valido = (soma_in == soma_out == total_e)
        return {
            "soma_in": soma_in,
            "soma_out": soma_out,
            "total_e": total_e,
            "valido": valido
        }

# Definição dos nós de processo da Fábrica de Paçoca (Grupo 7)
nos_processo = [
    "SILO-101_Amendoim",
    "SILO-102_Acucar",
    "MAN-301_Dosagem",
    "MOI-301A",
    "MOI-301B",
    "HOM-301_Massa",
    "MOE-302_Pulmao",
    "PRN-401_Prensa"
]

rede = GrafoTubulacao(nos_processo)

# Cadastro das tubulações e dutos com instrumentação ISA-5.1 e dimensões físicas
rede.adicionar_tubulacao("SILO-101_Amendoim", "MAN-301_Dosagem", 15.0, "XV-301", 3.0)
rede.adicionar_tubulacao("SILO-102_Acucar", "MAN-301_Dosagem", 12.0, "XV-302", 4.0)
rede.adicionar_tubulacao("MAN-301_Dosagem", "MOI-301A", 8.0, "XV-303A", 4.0)
rede.adicionar_tubulacao("MAN-301_Dosagem", "MOI-301B", 10.0, "XV-303B", 4.0)
rede.adicionar_tubulacao("MOI-301A", "HOM-301_Massa", 25.0, "XV-304A", 4.0)
rede.adicionar_tubulacao("MOI-301B", "HOM-301_Massa", 22.0, "XV-304B", 4.0)
rede.adicionar_tubulacao("HOM-301_Massa", "PRN-401_Prensa", 30.0, "XV-401", 6.0)
rede.adicionar_tubulacao("HOM-301_Massa", "MOE-302_Pulmao", 18.0, "XV-402", 6.0)
rede.adicionar_tubulacao("MOE-302_Pulmao", "PRN-401_Prensa", 20.0, "XV-403", 6.0)

# 1. Exibição da Tabela de Dutos de Processo
print("Tabela de Dutos de Processo (Fábrica de Paçoca):")
print(formatar_tabela(rede.arestas_detalhes))

# 2. Exibição da Tabela de Graus Topológicos
print("\n--- Graus Topológicos ---")
print(formatar_tabela(rede.obter_graus()))

# 3. Verificação do Lema do Aperto de Mãos Dirigido
aperto = rede.verificar_aperto_maos()
print("\n--- Verificação Formal: Lema do Aperto de Mãos Dirigido ---")
print(f"Soma dos Graus de Entrada (deg-): {aperto['soma_in']}")
print(f"Soma dos Graus de Saída (deg+):   {aperto['soma_out']}")
print(f"Total de Dutos Cadastrados (|E|): {aperto['total_e']}")
status_str = "APROVADO (Consistência Topológica Confirmada)" if aperto['valido'] else "REPROVADO (Inconsistência)"
print(f"Consistência Topológica:          {status_str}")

# 4. Exibição da Matriz de Adjacência Ponderada (Metros)
print("\n--- Matriz de Adjacência Ponderada (Metros) ---")
print(formatar_matriz(rede.adj_pesos, rede.vertices, rede.vertices))

# 5. Exibição da Matriz de Adjacência Binária
print("\n--- Matriz de Adjacência Binária (Conectividade Direta) ---")
print(formatar_matriz(rede.adj_binaria, rede.vertices, rede.vertices))


Tabela de Dutos de Processo (Fábrica de Paçoca):
Origem            | Destino         | Comprimento (m) | Válvula ISA | Diâmetro (pol)
------------------+-----------------+-----------------+-------------+---------------
SILO-101_Amendoim | MAN-301_Dosagem | 15.0            | XV-301      | 3.0           
SILO-102_Acucar   | MAN-301_Dosagem | 12.0            | XV-302      | 4.0           
MAN-301_Dosagem   | MOI-301A        | 8.0             | XV-303A     | 4.0           
MAN-301_Dosagem   | MOI-301B        | 10.0            | XV-303B     | 4.0           
MOI-301A          | HOM-301_Massa   | 25.0            | XV-304A     | 4.0           
MOI-301B          | HOM-301_Massa   | 22.0            | XV-304B     | 4.0           
HOM-301_Massa     | PRN-401_Prensa  | 30.0            | XV-401      | 6.0           
HOM-301_Massa     | MOE-302_Pulmao  | 18.0            | XV-402      | 6.0           
MOE-302_Pulmao    | PRN-401_Prensa  | 20.0            | XV-403      | 6.0           

--- Graus Topol